# Full Model Training Pipeline

This notebook trains **9 XGBoost models** - one for each combination of:
- **Tag Groups**: Genre, Mood, Situation
- **Optimization Metrics**: Precision, Recall, F1

## Pipeline Steps
1. Load training dataset
2. Hyperparameter tuning (100 random trials per model)
3. Threshold optimization per metric
4. Performance comparison across tag groups
5. Save all trained models

In [1]:
# Setup and imports
%load_ext autoreload
%autoreload 2

from pyrekordbox import Rekordbox6Database
import polars as pl
import numpy as np
from nbutils import setup_path, display_polars

setup_path()

from utils import get_base_dataset
from processing import preprocess_tag_group
from models import (
    tune_xgboost_hyperparams,
    optimize_prediction_thresholds,
    save_model,
)

db = Rekordbox6Database()
pl.Config.set_tbl_rows(30)

[10:32:18] pyrekordbox.db6.database:WARNING  - Rekordbox is running!


polars.config.Config

## 1. Data Loading and Preparation

Load the base dataset and audio features, then prepare train/validation/test splits.

In [2]:
# Get base dataset
base_df = get_base_dataset(db, min_tag_count=5)

# Load audio features
features_df = pl.read_parquet("../data/song_features.parquet")

# Define feature columns to exclude
exclude_cols = [
    "song_path",
    "harmonic_percussive_ratio",
    "percussive_strength",
    "tonnetz_mean_0", "tonnetz_mean_1", "tonnetz_mean_2", "tonnetz_mean_3", "tonnetz_mean_4", "tonnetz_mean_5",
    "tonnetz_std_0", "tonnetz_std_1", "tonnetz_std_2", "tonnetz_std_3", "tonnetz_std_4", "tonnetz_std_5"
]

feature_cols = [col for col in features_df.columns 
                if col not in exclude_cols + ["song_id", "song_path"]]

# Filter features to remove null values
features_df = features_df.filter(pl.col("energy_increase_ratio").is_not_null())

print(f"\nFeature matrix shape: {features_df.shape}")
print(f"Number of features: {len(feature_cols)}")


Filtering tags with fewer than 5 occurrences:

Genre:
  - Grime: 1 occurrence(s)
  - New Beat: 3 occurrence(s)
  - Dancehall: 3 occurrence(s)
  - Blues: 4 occurrence(s)
  - Gabber: 4 occurrence(s)

Mood:
  - Industrial: 1 occurrence(s)

Total tags filtered: 6


Feature matrix shape: (515, 376)
Number of features: 360


## 2. Train Models for All Tag Groups and Metrics

We'll train 9 models total:
- Genre: precision, recall, f1
- Mood: precision, recall, f1  
- Situation: precision, recall, f1

In [3]:
# Configuration
TAG_GROUPS = ["Genre", "Mood", "Situation"]
METRICS = ["precision", "recall", "f1"]
N_TRIALS = 100  # Random search iterations per model

# Store all results
all_models = {}
all_threshold_results = {}
all_preprocessed_data = {}

print(f"Training {len(TAG_GROUPS)} × {len(METRICS)} = {len(TAG_GROUPS) * len(METRICS)} models")
print(f"Each model: {N_TRIALS} hyperparameter trials + threshold optimization")

Training 3 × 3 = 9 models
Each model: 100 hyperparameter trials + threshold optimization


### 2.1 Genre Models

In [4]:
# Preprocess Genre data
print("="*80)
print("PREPROCESSING: GENRE")
print("="*80)

genre_result = preprocess_tag_group(
    base_df, features_df, "Genre",
    feature_cols=feature_cols,
    test_size=0.2,
    val_size=0.2,
    min_train_count=10,
    apply_scaling=True,
    apply_pca=False,
)

all_preprocessed_data["Genre"] = genre_result

print(f"\nGenre - Data split:")
print(f"  Train: {genre_result.X_train.shape[0]} samples")
print(f"  Validation: {genre_result.X_val.shape[0]} samples")
print(f"  Test: {genre_result.X_test.shape[0]} samples")
print(f"  Labels: {len(genre_result.tags)}")

PREPROCESSING: GENRE

PREPROCESSING: GENRE

Preparing multi-label data for 'Genre':
Unique songs: 696
Unique tags: 43
Tags: Acid, Afrobeat, Ambient, Ballad, Bass, Beats, Boogie, Breakbeat, Disco, Drum & Bass, Dubstep, Electro, Eurodance, Folk, World & Country, Footwork, Funk, Future Bass, Garage, Hip-Hop, House, Indie, Jazz, Juke, Jungle, Motown, Neo Soul, New Wave, Old School, Pop, Punk, R&B, Rap, Reggae, Reggaeton, Rock, Rock & Roll, Singer-Songwriter, Soul, Techno, Trance, Tribal, Trip-Hop, Yaught Rock

Final dataset shape:
  X: (499, 361) (song_id + 360 features)
  y: (499, 43) (43 binary labels)
  Average tags per song: 3.48


Genre - Initial label statistics:
shape: (43, 3)
┌───────────────────┬───────┬────────────┐
│ tag               ┆ count ┆ percentage │
│ ---               ┆ ---   ┆ ---        │
│ str               ┆ i64   ┆ f64        │
╞═══════════════════╪═══════╪════════════╡
│ Soul              ┆ 266   ┆ 53.306613  │
│ House             ┆ 204   ┆ 40.881764  │
│ Funk    

In [5]:
# Genre - Precision Model
print("\n" + "="*80)
print("GENRE - PRECISION MODEL")
print("="*80)

# Hyperparameter tuning optimized for precision
genre_precision_results = tune_xgboost_hyperparams(
    X_train=genre_result.X_train,
    y_train=genre_result.y_train,
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='precision',
    verbose=True
)

# Threshold optimization for precision
genre_precision_thresholds = optimize_prediction_thresholds(
    model=genre_precision_results['best_model'],
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    metric='precision',
    verbose=True
)

# Store results
all_models['genre_precision'] = genre_precision_results
all_threshold_results['genre_precision'] = genre_precision_thresholds


GENRE - PRECISION MODEL

XGBOOST HYPERPARAMETER TUNING
Search type: random
Optimization metric: precision

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best precision score: 0.5598

Best parameters:
  n_estimators: 200
  max_depth: 3
  learn

/Users/quintenrosseel/Development/personal/music_tagger/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [6]:
# Genre - Recall Model
print("\n" + "="*80)
print("GENRE - RECALL MODEL")
print("="*80)

# Hyperparameter tuning optimized for recall
genre_recall_results = tune_xgboost_hyperparams(
    X_train=genre_result.X_train,
    y_train=genre_result.y_train,
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='recall',
    verbose=True
)

# Threshold optimization for recall
genre_recall_thresholds = optimize_prediction_thresholds(
    model=genre_recall_results['best_model'],
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    metric='recall',
    verbose=True
)

# Store results
all_models['genre_recall'] = genre_recall_results
all_threshold_results['genre_recall'] = genre_recall_thresholds


GENRE - RECALL MODEL

XGBOOST HYPERPARAMETER TUNING
Search type: random
Optimization metric: recall

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best recall score: 0.4674

Best parameters:
  n_estimators: 200
  max_depth: 3
  learning_rate:

/Users/quintenrosseel/Development/personal/music_tagger/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [7]:
# Genre - F1 Model
print("\n" + "="*80)
print("GENRE - F1 MODEL")
print("="*80)

# Hyperparameter tuning optimized for F1
genre_f1_results = tune_xgboost_hyperparams(
    X_train=genre_result.X_train,
    y_train=genre_result.y_train,
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='f1',
    verbose=True
)

# Threshold optimization for f1
genre_f1_thresholds = optimize_prediction_thresholds(
    model=genre_f1_results['best_model'],
    X_val=genre_result.X_val,
    y_val=genre_result.y_val,
    tags=genre_result.tags,
    metric='f1',
    verbose=True
)

# Store results
all_models['genre_f1'] = genre_f1_results
all_threshold_results['genre_f1'] = genre_f1_thresholds


GENRE - F1 MODEL

XGBOOST HYPERPARAMETER TUNING
Search type: random
Optimization metric: f1

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best f1 score: 0.3687

Best parameters:
  n_estimators: 100
  max_depth: 3
  learning_rate: 0.05
  subs

/Users/quintenrosseel/Development/personal/music_tagger/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


### 2.2 Mood Models

In [8]:
# Preprocess Mood data
print("="*80)
print("PREPROCESSING: MOOD")
print("="*80)

mood_result = preprocess_tag_group(
    base_df, features_df, "Mood",
    feature_cols=feature_cols,
    test_size=0.2,
    val_size=0.2,
    min_train_count=10,
    apply_scaling=True,
    apply_pca=False,
)

all_preprocessed_data["Mood"] = mood_result

print(f"\nMood - Data split:")
print(f"  Train: {mood_result.X_train.shape[0]} samples")
print(f"  Validation: {mood_result.X_val.shape[0]} samples")
print(f"  Test: {mood_result.X_test.shape[0]} samples")
print(f"  Labels: {len(mood_result.tags)}")

PREPROCESSING: MOOD

PREPROCESSING: MOOD

Preparing multi-label data for 'Mood':
Unique songs: 688
Unique tags: 23
Tags: Chill, Dancefloor, Dark, Deep, Experimental, Good Vibes, Groovy, Guilty, Hippy, Like a boss, Love, Minimal, Mysterious, Pretty, Sad, Spacy, Stadium, Sunset, Synths, Temposhifter, Trippy, Uplifting, Uptempo

Final dataset shape:
  X: (496, 361) (song_id + 360 features)
  y: (496, 23) (23 binary labels)
  Average tags per song: 4.33


Mood - Initial label statistics:
shape: (23, 3)
┌──────────────┬───────┬────────────┐
│ tag          ┆ count ┆ percentage │
│ ---          ┆ ---   ┆ ---        │
│ str          ┆ i64   ┆ f64        │
╞══════════════╪═══════╪════════════╡
│ Good Vibes   ┆ 281   ┆ 56.653226  │
│ Dancefloor   ┆ 278   ┆ 56.048387  │
│ Uplifting    ┆ 259   ┆ 52.217742  │
│ Sunset       ┆ 231   ┆ 46.572581  │
│ Groovy       ┆ 211   ┆ 42.540323  │
│ Deep         ┆ 133   ┆ 26.814516  │
│ Chill        ┆ 123   ┆ 24.798387  │
│ Pretty       ┆ 112   ┆ 22.580645  │
│ 

In [9]:
# Mood - Precision Model
print("\n" + "="*80)
print("MOOD - PRECISION MODEL")
print("="*80)

mood_precision_results = tune_xgboost_hyperparams(
    X_train=mood_result.X_train,
    y_train=mood_result.y_train,
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='precision',
    verbose=True
)

mood_precision_thresholds = optimize_prediction_thresholds(
    model=mood_precision_results['best_model'],
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    metric='precision',
    verbose=True
)

all_models['mood_precision'] = mood_precision_results
all_threshold_results['mood_precision'] = mood_precision_thresholds


MOOD - PRECISION MODEL

XGBOOST HYPERPARAMETER TUNING
Search type: random
Optimization metric: precision

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best precision score: 0.4782

Best parameters:
  n_estimators: 300
  max_depth: 15
  learn

/Users/quintenrosseel/Development/personal/music_tagger/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [10]:
# Mood - Recall Model
print("\n" + "="*80)
print("MOOD - RECALL MODEL")
print("="*80)

mood_recall_results = tune_xgboost_hyperparams(
    X_train=mood_result.X_train,
    y_train=mood_result.y_train,
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='recall',
    verbose=True
)

mood_recall_thresholds = optimize_prediction_thresholds(
    model=mood_recall_results['best_model'],
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    metric='recall',
    verbose=True
)

all_models['mood_recall'] = mood_recall_results
all_threshold_results['mood_recall'] = mood_recall_thresholds


MOOD - RECALL MODEL

XGBOOST HYPERPARAMETER TUNING
Search type: random
Optimization metric: recall

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best recall score: 0.4910

Best parameters:
  n_estimators: 200
  max_depth: 3
  learning_rate: 

/Users/quintenrosseel/Development/personal/music_tagger/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


In [11]:
# Mood - F1 Model
print("\n" + "="*80)
print("MOOD - F1 MODEL")
print("="*80)

mood_f1_results = tune_xgboost_hyperparams(
    X_train=mood_result.X_train,
    y_train=mood_result.y_train,
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='f1',
    verbose=True
)

mood_f1_thresholds = optimize_prediction_thresholds(
    model=mood_f1_results['best_model'],
    X_val=mood_result.X_val,
    y_val=mood_result.y_val,
    tags=mood_result.tags,
    metric='f1',
    verbose=True
)

all_models['mood_f1'] = mood_f1_results
all_threshold_results['mood_f1'] = mood_f1_thresholds


MOOD - F1 MODEL

XGBOOST HYPERPARAMETER TUNING
Search type: random
Optimization metric: f1

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best f1 score: 0.3751

Best parameters:
  n_estimators: 100
  max_depth: 3
  learning_rate: 0.1
  subsam

/Users/quintenrosseel/Development/personal/music_tagger/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


### 2.3 Situation Models

In [12]:
# Preprocess Situation data
print("="*80)
print("PREPROCESSING: SITUATION")
print("="*80)

situation_result = preprocess_tag_group(
    base_df, features_df, "Situation",
    feature_cols=feature_cols,
    test_size=0.2,
    val_size=0.2,
    min_train_count=10,
    apply_scaling=True,
    apply_pca=False,
)

all_preprocessed_data["Situation"] = situation_result

print(f"\nSituation - Data split:")
print(f"  Train: {situation_result.X_train.shape[0]} samples")
print(f"  Validation: {situation_result.X_val.shape[0]} samples")
print(f"  Test: {situation_result.X_test.shape[0]} samples")
print(f"  Labels: {len(situation_result.tags)}")

PREPROCESSING: SITUATION

PREPROCESSING: SITUATION

Preparing multi-label data for 'Situation':
Unique songs: 737
Unique tags: 20
Tags: After Hours, Background, Build up, Burning Man, Classics, Dirty, Favs, Ladies, Lisa & Ant Daytime, Lisa & Ant Nighttime, Live Instrument Gems, Loungy, Matiné, Peak, Rave, Sing Along, Skank, Surfing, Twerk, Twist

Final dataset shape:
  X: (500, 361) (song_id + 360 features)
  y: (500, 20) (20 binary labels)
  Average tags per song: 4.63


Situation - Initial label statistics:
shape: (20, 3)
┌──────────────────────┬───────┬────────────┐
│ tag                  ┆ count ┆ percentage │
│ ---                  ┆ ---   ┆ ---        │
│ str                  ┆ i64   ┆ f64        │
╞══════════════════════╪═══════╪════════════╡
│ Build up             ┆ 237   ┆ 47.4       │
│ Live Instrument Gems ┆ 210   ┆ 42.0       │
│ Peak                 ┆ 206   ┆ 41.2       │
│ Loungy               ┆ 203   ┆ 40.6       │
│ After Hours          ┆ 202   ┆ 40.4       │
│ Favs    

In [13]:
# Situation - Precision Model
print("\n" + "="*80)
print("SITUATION - PRECISION MODEL")
print("="*80)

situation_precision_results = tune_xgboost_hyperparams(
    X_train=situation_result.X_train,
    y_train=situation_result.y_train,
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='precision',
    verbose=True
)

situation_precision_thresholds = optimize_prediction_thresholds(
    model=situation_precision_results['best_model'],
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    metric='precision',
    verbose=True
)

all_models['situation_precision'] = situation_precision_results
all_threshold_results['situation_precision'] = situation_precision_thresholds


SITUATION - PRECISION MODEL

XGBOOST HYPERPARAMETER TUNING
Search type: random
Optimization metric: precision

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best precision score: 0.5254

Best parameters:
  n_estimators: 200
  max_depth: 3
  l

In [14]:
# Situation - Recall Model
print("\n" + "="*80)
print("SITUATION - RECALL MODEL")
print("="*80)

situation_recall_results = tune_xgboost_hyperparams(
    X_train=situation_result.X_train,
    y_train=situation_result.y_train,
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='recall',
    verbose=True
)

situation_recall_thresholds = optimize_prediction_thresholds(
    model=situation_recall_results['best_model'],
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    metric='recall',
    verbose=True
)

all_models['situation_recall'] = situation_recall_results
all_threshold_results['situation_recall'] = situation_recall_thresholds


SITUATION - RECALL MODEL

XGBOOST HYPERPARAMETER TUNING
Search type: random
Optimization metric: recall

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best recall score: 0.6370

Best parameters:
  n_estimators: 100
  max_depth: 3
  learning_r

In [15]:
# Situation - F1 Model
print("\n" + "="*80)
print("SITUATION - F1 MODEL")
print("="*80)

situation_f1_results = tune_xgboost_hyperparams(
    X_train=situation_result.X_train,
    y_train=situation_result.y_train,
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    search_type='random',
    n_iter=N_TRIALS,
    metric='f1',
    verbose=True
)

situation_f1_thresholds = optimize_prediction_thresholds(
    model=situation_f1_results['best_model'],
    X_val=situation_result.X_val,
    y_val=situation_result.y_val,
    tags=situation_result.tags,
    metric='f1',
    verbose=True
)

all_models['situation_f1'] = situation_f1_results
all_threshold_results['situation_f1'] = situation_f1_thresholds


SITUATION - F1 MODEL

XGBOOST HYPERPARAMETER TUNING
Search type: random
Optimization metric: f1

Testing 100 random parameter combinations...

Progress: 5/100 combinations tested
Progress: 10/100 combinations tested
Progress: 15/100 combinations tested
Progress: 20/100 combinations tested
Progress: 25/100 combinations tested
Progress: 30/100 combinations tested
Progress: 35/100 combinations tested
Progress: 40/100 combinations tested
Progress: 45/100 combinations tested
Progress: 50/100 combinations tested
Progress: 55/100 combinations tested
Progress: 60/100 combinations tested
Progress: 65/100 combinations tested
Progress: 70/100 combinations tested
Progress: 75/100 combinations tested
Progress: 80/100 combinations tested
Progress: 85/100 combinations tested
Progress: 90/100 combinations tested
Progress: 95/100 combinations tested
Progress: 100/100 combinations tested

TUNING RESULTS

Best f1 score: 0.4072

Best parameters:
  n_estimators: 100
  max_depth: 10
  learning_rate: 0.1
  

## 3. Performance Comparison

Compare all models across tag groups and optimization metrics.

In [16]:
# Create comprehensive comparison table
comparison_data = []

for model_key, threshold_result in all_threshold_results.items():
    tag_group, metric = model_key.rsplit('_', 1)
    
    # Get preprocessed data for this tag group
    tag_group_name = tag_group.capitalize()
    data = all_preprocessed_data[tag_group_name]
    
    comparison_data.append({
        "tag_group": tag_group_name,
        "optimization_metric": metric,
        "model_name": f"xgboost_{tag_group}_{metric}",
        "n_tags": len(data.tags),
        "tags": ", ".join(data.tags),
        "train_support": data.X_train.shape[0],
        "val_support": data.X_val.shape[0],
        "test_support": data.X_test.shape[0],
        "macro_precision": threshold_result['optimized_metrics']['macro_precision'],
        "macro_recall": threshold_result['optimized_metrics']['macro_recall'],
        "macro_f1": threshold_result['optimized_metrics']['macro_f1'],
        "weighted_precision": threshold_result['optimized_metrics']['weighted_precision'],
        "weighted_recall": threshold_result['optimized_metrics']['weighted_recall'],
        "weighted_f1": threshold_result['optimized_metrics']['weighted_f1'],
        "default_f1": threshold_result['default_metrics']['macro_f1'],
        "f1_improvement": threshold_result['improvement'],
    })

comparison_df = pl.DataFrame(comparison_data)

print("\n" + "="*100)
print("MODEL PERFORMANCE COMPARISON")
print("="*100)
print("\nAll 9 Models - Sorted by Tag Group and Optimization Metric:")
display_polars(comparison_df.sort(["tag_group", "optimization_metric"]), lim=20)


MODEL PERFORMANCE COMPARISON

All 9 Models - Sorted by Tag Group and Optimization Metric:
shape: (9, 16)
┌───────────┬────────────┬────────────┬────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ tag_group ┆ optimizati ┆ model_name ┆ n_tags ┆ … ┆ weighted_ ┆ weighted_ ┆ default_f ┆ f1_improv │
│ ---       ┆ on_metric  ┆ ---        ┆ ---    ┆   ┆ recall    ┆ f1        ┆ 1         ┆ ement     │
│ str       ┆ ---        ┆ str        ┆ i64    ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│           ┆ str        ┆            ┆        ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞═══════════╪════════════╪════════════╪════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Genre     ┆ f1         ┆ xgboost_ge ┆ 22     ┆ … ┆ 0.745342  ┆ 0.596953  ┆ 0.368652  ┆ 0.136802  │
│           ┆            ┆ nre_f1     ┆        ┆   ┆           ┆           ┆           ┆           │
│ Genre     ┆ precision  ┆ xgboost_ge ┆ 22     ┆ … ┆ 0.170807  ┆ 0.265236  ┆ 0.256087 

In [17]:
# Compare by optimization metric
print("\n" + "="*100)
print("PERFORMANCE BY OPTIMIZATION METRIC")
print("="*100)

for metric in METRICS:
    print(f"\n{metric.upper()} Models:")
    metric_models = comparison_df.filter(pl.col("optimization_metric") == metric)
    display_polars(
        metric_models.select([
            "tag_group", "n_tags", 
            "macro_precision", "macro_recall", "macro_f1",
            "weighted_precision", "weighted_recall", "weighted_f1",
            "f1_improvement"
        ]).sort("macro_f1", descending=True),
        lim=10
    )


PERFORMANCE BY OPTIMIZATION METRIC

PRECISION Models:
shape: (3, 9)
┌───────────┬────────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ tag_group ┆ n_tags ┆ macro_prec ┆ macro_reca ┆ … ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ f1_improv │
│ ---       ┆ ---    ┆ ision      ┆ ll         ┆   ┆ precision ┆ recall    ┆ f1        ┆ ement     │
│ str       ┆ i64    ┆ ---        ┆ ---        ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│           ┆        ┆ f64        ┆ f64        ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞═══════════╪════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Genre     ┆ 22     ┆ 0.704679   ┆ 0.148854   ┆ … ┆ 0.855179  ┆ 0.170807  ┆ 0.265236  ┆ -0.025987 │
│ Mood      ┆ 19     ┆ 0.643445   ┆ 0.187119   ┆ … ┆ 0.797076  ┆ 0.272973  ┆ 0.288843  ┆ -0.120606 │
│ Situation ┆ 19     ┆ 0.721937   ┆ 0.169885   ┆ … ┆ 0.793066  ┆ 0.160752  ┆ 0.213485  ┆ -0.190665 │
└───────────┴────────┴

In [18]:
# Compare by tag group
print("\n" + "="*100)
print("PERFORMANCE BY TAG GROUP")
print("="*100)

for tag_group in TAG_GROUPS:
    print(f"\n{tag_group.upper()} Models:")
    group_models = comparison_df.filter(pl.col("tag_group") == tag_group)
    display_polars(
        group_models.select([
            "optimization_metric", "n_tags",
            "macro_precision", "macro_recall", "macro_f1",
            "weighted_precision", "weighted_recall", "weighted_f1",
            "f1_improvement"
        ]).sort("optimization_metric"),
        lim=10
    )


PERFORMANCE BY TAG GROUP

GENRE Models:
shape: (3, 9)
┌────────────┬────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ optimizati ┆ n_tags ┆ macro_prec ┆ macro_rec ┆ … ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ f1_improv │
│ on_metric  ┆ ---    ┆ ision      ┆ all       ┆   ┆ precision ┆ recall    ┆ f1        ┆ ement     │
│ ---        ┆ i64    ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆        ┆ f64        ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ f1         ┆ 22     ┆ 0.518072   ┆ 0.59847   ┆ … ┆ 0.546612  ┆ 0.745342  ┆ 0.596953  ┆ 0.136802  │
│ precision  ┆ 22     ┆ 0.704679   ┆ 0.148854  ┆ … ┆ 0.855179  ┆ 0.170807  ┆ 0.265236  ┆ -0.025987 │
│ recall     ┆ 22     ┆ 0.146364   ┆ 0.954545  ┆ … ┆ 0.27      ┆ 1.0       ┆ 0.394978  ┆ -0.138493 │
└────────────┴────────┴────────────┴

In [19]:
# Show best model per tag group
print("\n" + "="*100)
print("BEST MODEL PER TAG GROUP (by Macro F1 Score)")
print("="*100)

best_models = (
    comparison_df
    .sort("macro_f1", descending=True)
    .group_by("tag_group")
    .first()
    .sort("tag_group")
)

display_polars(
    best_models.select([
        "tag_group", "optimization_metric", "n_tags",
        "macro_precision", "macro_recall", "macro_f1",
        "weighted_precision", "weighted_recall", "weighted_f1"
    ]),
    lim=10
)


BEST MODEL PER TAG GROUP (by Macro F1 Score)
shape: (3, 9)
┌───────────┬────────────┬────────┬────────────┬───┬──────────┬────────────┬───────────┬───────────┐
│ tag_group ┆ optimizati ┆ n_tags ┆ macro_prec ┆ … ┆ macro_f1 ┆ weighted_p ┆ weighted_ ┆ weighted_ │
│ ---       ┆ on_metric  ┆ ---    ┆ ision      ┆   ┆ ---      ┆ recision   ┆ recall    ┆ f1        │
│ str       ┆ ---        ┆ i64    ┆ ---        ┆   ┆ f64      ┆ ---        ┆ ---       ┆ ---       │
│           ┆ str        ┆        ┆ f64        ┆   ┆          ┆ f64        ┆ f64       ┆ f64       │
╞═══════════╪════════════╪════════╪════════════╪═══╪══════════╪════════════╪═══════════╪═══════════╡
│ Genre     ┆ f1         ┆ 22     ┆ 0.518072   ┆ … ┆ 0.505454 ┆ 0.546612   ┆ 0.745342  ┆ 0.596953  │
│ Mood      ┆ f1         ┆ 19     ┆ 0.405388   ┆ … ┆ 0.453251 ┆ 0.478361   ┆ 0.824324  ┆ 0.587198  │
│ Situation ┆ f1         ┆ 19     ┆ 0.453563   ┆ … ┆ 0.511909 ┆ 0.492523   ┆ 0.791232  ┆ 0.585897  │
└───────────┴────────────┴─────

## 4. Save All Models

Save all 9 trained models with their configurations, thresholds, and preprocessing components.

In [20]:
# Save all models
print("\n" + "="*100)
print("SAVING ALL MODELS")
print("="*100)

saved_paths = {}

for model_key in all_models.keys():
    tag_group, metric = model_key.rsplit('_', 1)
    tag_group_name = tag_group.capitalize()
    
    # Get data
    data = all_preprocessed_data[tag_group_name]
    model_results = all_models[model_key]
    threshold_results = all_threshold_results[model_key]
    
    # Save model with both macro and weighted metrics
    paths = save_model(
        model=model_results['best_model'],
        model_name=f"xgboost_{tag_group_name}_{metric}",
        save_dir="../models",
        tags=data.tags,
        thresholds=threshold_results['thresholds'],
        hyperparams=model_results['best_params'],
        scaler=data.scaler,
        pca=data.pca,
        metrics={
            "macro_precision": threshold_results['optimized_metrics']['macro_precision'],
            "macro_recall": threshold_results['optimized_metrics']['macro_recall'],
            "macro_f1": threshold_results['optimized_metrics']['macro_f1'],
            "weighted_precision": threshold_results['optimized_metrics']['weighted_precision'],
            "weighted_recall": threshold_results['optimized_metrics']['weighted_recall'],
            "weighted_f1": threshold_results['optimized_metrics']['weighted_f1'],
            "optimization_metric": metric,
        },
        tag_group=tag_group_name,
        verbose=True
    )
    
    saved_paths[model_key] = paths

print("\n" + "="*100)
print(f"SUCCESSFULLY SAVED {len(saved_paths)} MODELS")
print("="*100)


SAVING ALL MODELS

MODEL SAVED SUCCESSFULLY
Model name: xgboost_Genre_precision
Tag group: Genre
Number of labels: 22

Files saved:
  Model:         ../models/xgboost_Genre_precision_20251018_111519_model.pkl
  Configuration: ../models/xgboost_Genre_precision_20251018_111519_config.json
  Thresholds:    ../models/xgboost_Genre_precision_20251018_111519_thresholds.npy
  Scaler:        ../models/xgboost_Genre_precision_20251018_111519_scaler.pkl

Metrics:
  macro_precision: 0.7047
  macro_recall: 0.1489
  macro_f1: 0.2301
  weighted_precision: 0.8552
  weighted_recall: 0.1708
  weighted_f1: 0.2652
  optimization_metric: precision


MODEL SAVED SUCCESSFULLY
Model name: xgboost_Genre_recall
Tag group: Genre
Number of labels: 22

Files saved:
  Model:         ../models/xgboost_Genre_recall_20251018_111519_model.pkl
  Configuration: ../models/xgboost_Genre_recall_20251018_111519_config.json
  Thresholds:    ../models/xgboost_Genre_recall_20251018_111519_thresholds.npy
  Scaler:        ../mo

In [21]:
# Summary of saved models
print("\nSaved Model Files:")
for model_key, paths in saved_paths.items():
    print(f"\n{model_key}:")
    for file_type, path in paths.items():
        if path:
            print(f"  {file_type}: {path}")


Saved Model Files:

genre_precision:
  model: ../models/xgboost_Genre_precision_20251018_111519_model.pkl
  config: ../models/xgboost_Genre_precision_20251018_111519_config.json
  thresholds: ../models/xgboost_Genre_precision_20251018_111519_thresholds.npy
  scaler: ../models/xgboost_Genre_precision_20251018_111519_scaler.pkl

genre_recall:
  model: ../models/xgboost_Genre_recall_20251018_111519_model.pkl
  config: ../models/xgboost_Genre_recall_20251018_111519_config.json
  thresholds: ../models/xgboost_Genre_recall_20251018_111519_thresholds.npy
  scaler: ../models/xgboost_Genre_recall_20251018_111519_scaler.pkl

genre_f1:
  model: ../models/xgboost_Genre_f1_20251018_111519_model.pkl
  config: ../models/xgboost_Genre_f1_20251018_111519_config.json
  thresholds: ../models/xgboost_Genre_f1_20251018_111519_thresholds.npy
  scaler: ../models/xgboost_Genre_f1_20251018_111519_scaler.pkl

mood_precision:
  model: ../models/xgboost_Mood_precision_20251018_111519_model.pkl
  config: ../model

## 5. Training Summary

Final summary of all trained models and their performance.

In [22]:
print("\n" + "="*100)
print("TRAINING COMPLETE - FINAL SUMMARY")
print("="*100)

print(f"\nTotal models trained: {len(all_models)}")
print(f"Tag groups: {', '.join(TAG_GROUPS)}")
print(f"Optimization metrics: {', '.join(METRICS)}")
print(f"Hyperparameter trials per model: {N_TRIALS}")

print("\nPerformance Summary (Macro Averages):")
summary_stats = comparison_df.select([
    pl.col("macro_f1").mean().alias("avg_macro_f1"),
    pl.col("macro_f1").max().alias("best_macro_f1"),
    pl.col("macro_precision").mean().alias("avg_macro_precision"),
    pl.col("macro_recall").mean().alias("avg_macro_recall"),
    pl.col("weighted_f1").mean().alias("avg_weighted_f1"),
    pl.col("weighted_precision").mean().alias("avg_weighted_precision"),
    pl.col("weighted_recall").mean().alias("avg_weighted_recall"),
    pl.col("f1_improvement").mean().alias("avg_f1_improvement"),
])

print(summary_stats)

print("\nBest Overall Model (by Macro F1):")
best_overall = comparison_df.sort("macro_f1", descending=True).head(1)
display_polars(
    best_overall.select([
        "model_name", "tag_group", "optimization_metric",
        "macro_precision", "macro_recall", "macro_f1",
        "weighted_precision", "weighted_recall", "weighted_f1"
    ]),
    lim=5
)

print("\n" + "="*100)
print("All models saved to: ../models/")
print("="*100)


TRAINING COMPLETE - FINAL SUMMARY

Total models trained: 9
Tag groups: Genre, Mood, Situation
Optimization metrics: precision, recall, f1
Hyperparameter trials per model: 100

Performance Summary (Macro Averages):
shape: (1, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ avg_macro_ ┆ best_macro ┆ avg_macro_ ┆ avg_macro ┆ avg_weigh ┆ avg_weigh ┆ avg_weigh ┆ avg_f1_im │
│ f1         ┆ _f1        ┆ precision  ┆ _recall   ┆ ted_f1    ┆ ted_preci ┆ ted_recal ┆ provement │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ sion      ┆ l         ┆ ---       │
│ f64        ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ ---       ┆ ---       ┆ f64       │
│            ┆            ┆            ┆           ┆           ┆ f64       ┆ f64       ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0.341079   ┆ 0.511909   ┆ 0.44914    ┆ 0.596367  ┆ 0.434369  ┆